# Olist 电商数据探索与清洗


## 1. 环境准备 & 加载数据

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


from pathlib import Path
import os

当前目录 = Path.cwd().resolve()
项目目录 = 当前目录.parent if 当前目录.name == "notebooks" else 当前目录
数据库路径 = 项目目录 / "data" / "processed" / "olist.duckdb"
输出目录 = 项目目录 / "outputs" / "figures"
输出目录.mkdir(parents=True, exist_ok=True)
os.chdir(项目目录)

if not 数据库路径.exists():
    raise FileNotFoundError("找不到数据库，请先运行 scripts/00_setup_database.py")
con = duckdb.connect(str(数据库路径), read_only=True)


## 2. 各表概览

In [ ]:
# customers
print("=== customers ===")
df = con.execute("SELECT * FROM customers LIMIT 5").fetchdf()
display(df)

# orders
print()
print("=== orders (前5行) ===")
df = con.execute("SELECT * FROM orders LIMIT 5").fetchdf()
display(df)

In [ ]:
# order_items
print("=== order_items ===")
df = con.execute("SELECT * FROM order_items LIMIT 5").fetchdf()
display(df)

# order_payments
print()
print("=== order_payments ===")
df = con.execute("SELECT * FROM order_payments LIMIT 5").fetchdf()
display(df)

In [ ]:
# products
print("=== products ===")
df = con.execute("SELECT * FROM products LIMIT 5").fetchdf()
display(df)

# sellers
print()
print("=== sellers ===")
df = con.execute("SELECT * FROM sellers LIMIT 5").fetchdf()
display(df)

## 3. 缺失值检查

In [ ]:
def check_missing(con, table_name):
    cols = con.execute("SELECT column_name FROM information_schema.columns "
                       "WHERE table_name='" + table_name + "'").fetchdf()
    col_list = cols["column_name"].tolist()
    
    results = []
    for col in col_list:
        total = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        non_null = con.execute(f"SELECT COUNT({col}) FROM {table_name}").fetchone()[0]
        missing = total - non_null
        pct = round(missing / total * 100, 2) if total > 0 else 0
        results.append({"table": table_name, "column": col, "missing": missing, "pct": pct})
    
    return pd.DataFrame(results)

all_missing = pd.DataFrame()
for t in ["customers", "orders", "order_items", "order_payments", "products", "sellers"]:
    result = check_missing(con, t)
    all_missing = pd.concat([all_missing, result])

has_missing = all_missing[all_missing["missing"] > 0]
if len(has_missing) > 0:
    print(f"发现 {len(has_missing)} 个列有缺失值：")
    print(has_missing.to_string(index=False))
else:
    print("所有核心表均无缺失值！")

print()
print("=== orders 表时间字段检查 ===")
time_cols = ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]
for col in time_cols:
    missing = con.execute(f"SELECT COUNT(*) FROM orders WHERE {col} IS NULL").fetchone()[0]
    total = con.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
    pct = round(missing / total * 100, 2)
    print(f"  {col}: {missing}/{total} 缺失 ({pct}%)")

## 4. 重复值检查

In [ ]:
# customers
print("=== customers 唯一客户ID ===")
total_cust = con.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
unique_cust = con.execute("SELECT COUNT(DISTINCT customer_unique_id) FROM customers").fetchone()[0]
print(f"  总行数: {total_cust:,}")
print(f"  唯一客户: {unique_cust:,}")
print(f"  重复行: {total_cust - unique_cust:,}")

print()
print("=== orders：order_id 是否唯一？ ===")
total_ord = con.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
unique_ord = con.execute("SELECT COUNT(DISTINCT order_id) FROM orders").fetchone()[0]
print(f"  总行数: {total_ord:,}")
print(f"  唯一订单: {unique_ord:,}")

print()
print("=== 客户下单次数分布 ===")
order_cnt = con.execute("""
    SELECT order_count, COUNT(*) as customer_cnt
    FROM (
        SELECT customer_id, COUNT(*) as order_count
        FROM orders
        GROUP BY customer_id
    ) t
    GROUP BY order_count
    ORDER BY order_count
""").fetchdf()
print(order_cnt.head(10).to_string(index=False))
print(f"  最大下单次数: {order_cnt['order_count'].max()}")

## 5. 异常值检查

In [ ]:
print("=== 订单状态分布 ===")
status = con.execute("""
    SELECT order_status, COUNT(*) as cnt, 
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct
    FROM orders
    GROUP BY order_status
    ORDER BY cnt DESC
""").fetchdf()
print(status.to_string(index=False))

print()
print("=== 价格检查 ===")
zero_price = con.execute("SELECT COUNT(*) FROM order_items WHERE price <= 0").fetchone()[0]
print(f"  price <= 0 的订单行: {zero_price}")
max_price = con.execute("SELECT MAX(price) FROM order_items").fetchone()[0]
print(f"  price 最大值: {max_price:.2f}")

print()
print("=== 评价分数分布 ===")
scores = con.execute("""
    SELECT review_score, COUNT(*) as cnt
    FROM order_reviews
    GROUP BY review_score
    ORDER BY review_score
""").fetchdf()
print(scores.to_string(index=False))

## 6. 关键分布可视化

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 月订单量趋势
monthly_orders = con.execute("""
    SELECT strftime(order_purchase_timestamp, '%Y-%m') as month,
           COUNT(*) as order_count
    FROM orders
    GROUP BY month
    ORDER BY month
""").fetchdf()
axes[0, 0].plot(monthly_orders["month"].values, monthly_orders["order_count"].values, marker="o", linewidth=1.5)
axes[0, 0].tick_params(axis="x", rotation=45)
axes[0, 0].set_title("月订单量趋势")
axes[0, 0].set_ylabel("订单数")

# 价格分布
prices = con.execute("""
    SELECT price FROM order_items 
    WHERE price > 0 AND price < (SELECT PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY price) FROM order_items)
""").fetchdf()
axes[0, 1].hist(prices["price"].values, bins=50, edgecolor="white", alpha=0.7)
axes[0, 1].set_title("价格分布（去除99分位以上）")
axes[0, 1].set_xlabel("价格")
axes[0, 1].set_ylabel("频次")

# 评分分布
scores = con.execute("""
    SELECT review_score, COUNT(*) as cnt
    FROM order_reviews
    GROUP BY review_score
    ORDER BY review_score
""").fetchdf()
axes[0, 2].bar(scores["review_score"].values, scores["cnt"].values, color="steelblue", edgecolor="white")
axes[0, 2].set_title("评价分数分布")
axes[0, 2].set_xlabel("评分")
axes[0, 2].set_ylabel("数量")
for _, row in scores.iterrows():
    axes[0, 2].text(row["review_score"], row["cnt"] + 500, f"{row['cnt']:,}", ha="center", fontsize=9)

# 支付方式分布
payments = con.execute("""
    SELECT payment_type, COUNT(*) as cnt
    FROM order_payments
    GROUP BY payment_type
    ORDER BY cnt DESC
""").fetchdf()
# 小于 3% 的支付方式合并为“其他”，避免标签重叠
支付总数 = payments["cnt"].sum()
主要支付 = payments[payments["cnt"] / 支付总数 >= 0.03].copy()
其他支付数 = payments.loc[payments["cnt"] / 支付总数 < 0.03, "cnt"].sum()
if 其他支付数 > 0:
    主要支付 = pd.concat([主要支付, pd.DataFrame({"payment_type": ["其他"], "cnt": [其他支付数]})], ignore_index=True)
axes[1, 0].pie(主要支付["cnt"].values, labels=主要支付["payment_type"].values,
                autopct="%1.1f%%", startangle=90, labeldistance=1.12, pctdistance=0.68,
                wedgeprops={"edgecolor": "white", "linewidth": 1})
axes[1, 0].set_title("支付方式分布（小类合并）")

# 各州订单量 Top10
state_orders = con.execute("""
    SELECT c.customer_state, COUNT(*) as cnt
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_state
    ORDER BY cnt DESC
    LIMIT 10
""").fetchdf()
axes[1, 1].barh(state_orders["customer_state"].values[::-1], state_orders["cnt"].values[::-1], color="coral", edgecolor="white")
axes[1, 1].set_title("各州订单量 Top10")
axes[1, 1].set_xlabel("订单数")

# 商品类别 Top10
categories = con.execute("""
    SELECT ct.product_category_name_english, COUNT(*) as cnt
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN category_translation ct ON p.product_category_name = ct.product_category_name
    GROUP BY ct.product_category_name_english
    ORDER BY cnt DESC
    LIMIT 10
""").fetchdf()
axes[1, 2].barh(categories["product_category_name_english"].values[::-1], categories["cnt"].values[::-1], color="seagreen", edgecolor="white")
axes[1, 2].set_title("商品类别 Top10")
axes[1, 2].set_xlabel("销售数量")

plt.tight_layout()
plt.savefig(输出目录 / "eda_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("图表已保存到 outputs/figures/eda_distributions.png")

## 7. 时间范围确认

In [ ]:
time_range = con.execute("""
    SELECT MIN(order_purchase_timestamp) as first_order,
           MAX(order_purchase_timestamp) as last_order,
           DATEDIFF('day', MIN(order_purchase_timestamp), MAX(order_purchase_timestamp)) as days_span
    FROM orders
""").fetchdf()
print(f"首单日期: {time_range['first_order'][0]}")
print(f"末单日期: {time_range['last_order'][0]}")
print(f"覆盖天数: {time_range['days_span'][0]} 天")

## 8. 表关系验证

In [ ]:
items_orphan = con.execute("""
    SELECT COUNT(DISTINCT oi.order_id) 
    FROM order_items oi
    LEFT JOIN orders o ON oi.order_id = o.order_id
    WHERE o.order_id IS NULL
""").fetchone()[0]
print(f"  order_items → orders 找不到的ID: {items_orphan}")

pay_orphan = con.execute("""
    SELECT COUNT(DISTINCT op.order_id)
    FROM order_payments op
    LEFT JOIN orders o ON op.order_id = o.order_id
    WHERE o.order_id IS NULL
""").fetchone()[0]
print(f"  order_payments → orders 找不到的ID: {pay_orphan}")

prod_orphan = con.execute("""
    SELECT COUNT(DISTINCT oi.product_id)
    FROM order_items oi
    LEFT JOIN products p ON oi.product_id = p.product_id
    WHERE p.product_id IS NULL
""").fetchone()[0]
print(f"  order_items → products 找不到的ID: {prod_orphan}")

cust_orphan = con.execute("""
    SELECT COUNT(DISTINCT o.customer_id)
    FROM orders o
    LEFT JOIN customers c ON o.customer_id = c.customer_id
    WHERE c.customer_id IS NULL
""").fetchone()[0]
print(f"  orders → customers 找不到的ID: {cust_orphan}")

if items_orphan + pay_orphan + prod_orphan + cust_orphan == 0:
    print("\n 所有外键关系完整，数据质量良好！")
else:
    print("\n[WARN] 存在孤立记录，需进一步排查")

In [ ]:
con.close()
